In [ ]:
#after gpu is opened and dataset.rar is uploaded into the colab local environment (by draging and droping)

!unrar x "/content/dataset.rar" "/content/"

import numpy as np
import math
from typing import List
import os
import argparse
import glob
import shutil

def list_files(path):
    files = os.listdir(path)
    return np.asarray(files)

def split_files(oldpath, newpath, classes, train_validation_test):
    train_ratio = train_validation_test[0]
    validation_ratio = train_validation_test[1]
    test_ratio = train_validation_test[2]

    for name in classes:
        full_dir = os.path.join(os.getcwd(), f"{oldpath}/{name}")

        files = list_files(full_dir)
        total_file = np.size(files,0)
        # We split data set into 3: train, validation and test

        train_size = math.ceil(total_file * train_ratio)
        validation_size = train_size + math.ceil(total_file * validation_ratio)
        test_size = validation_size + math.ceil(total_file * test_ratio)

        train = files[0:train_size]
        validation = files[train_size:validation_size]
        test = files[validation_size:]

        move_files(train, full_dir, f"train/{name}")
        move_files(validation, full_dir, f"validation/{name}")
        move_files(test, full_dir, f"test/{name}")

def move_files(files, old_dir, new_dir):
    new_dir = os.path.join(os.getcwd(), new_dir);
    if not os.path.exists(new_dir):
        os.makedirs(new_dir)

    for file in np.nditer(files):
        old_file_path = os.path.join(os.getcwd(), f"{old_dir}/{file}")
        new_file_path = os.path.join(os.getcwd(), f"{new_dir}/{file}")

        shutil.move(old_file_path, new_file_path)

classes = ['notumor', 'meningioma', 'glioma', 'pituitary'];

split_files('dataset', './', classes, [0.7,0.2,0.1])


UNRAR 5.61 beta 1 freeware      Copyright (c) 1993-2018 Alexander Roshal

Cannot open /content/dataset.rar
No such file or directory
No files to extract


FileNotFoundError: ignored

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
#last part of resnet152 is .fc, whereas, last part of the efficientnet_b and densenet161 is .classifier[1]. Choose only one code block to use one of these pretrained models
#choose only one code block for desired pretrained model

# model = models.resnet152(pretrained=True)
# for param in model.parameters():
#   param.requires_grad = False
# fc_inputs = model.fc.in_features
# model.fc = nn.Sequential(
#     nn.Linear(fc_inputs, 1024),
#     nn.ReLU(inplace=True),

#     nn.Linear(1024, 4),
#     nn.Dropout(0.3),
#     nn.LogSoftmax(dim=1)
# )
# model = model.to(device)




model = models.densenet161(pretrained=True)
for param in model.parameters():
  param.requires_grad = False
classifier_inputs = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Linear(classifier_inputs, 1024),
    nn.ReLU(inplace=True),

    nn.Linear(1024, 4),
    nn.Dropout(0.3),
    nn.LogSoftmax(dim=1)
)
model = model.to(device)


#
# model = models.efficientnet_b3(pretrained=True)
# for param in model.parameters():
#   param.requires_grad = False
# classifier_inputs = model.classifier[1].in_features
# model.classifier[1] = nn.Sequential(
#     nn.Linear(classifier_inputs, 1024),
#     nn.ReLU(inplace=True),
#     nn.Linear(1024, 4),
#     nn.Dropout(0.3),
#     nn.LogSoftmax(dim=1)
# )
# model = model.to(device)

/usr/local/lib/python3.9/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.9/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet161_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet161_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet161-8d451a50.pth" to /root/.cache/torch/hub/checkpoints/densenet161-8d451a50.pth


  0%|          | 0.00/110M [00:00<?, ?B/s]

In [ ]:
from torchvision import transforms

# size1 and size2 values are dependent to the selected pre_trained _model, look at:

image_transforms = {
    'train': transforms.Compose([
              transforms.RandomRotation(degrees=20),
              transforms.Resize(size=256),
              transforms.RandomHorizontalFlip(p=0.5),
              transforms.RandomVerticalFlip(p=0.5),
              transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
              transforms.CenterCrop(size=224),
              transforms.ToTensor(),
              transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
    ]),
    'validation': transforms.Compose([
             transforms.Resize(size=256),
             transforms.CenterCrop(size=224),
             transforms.ToTensor(),
             transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
             transforms.Resize(size=256),
             transforms.CenterCrop(size=224),
             transforms.ToTensor(),
             transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
    ])
}

In [ ]:
import torchvision.datasets as datasets
import torch.utils.data as loader

data = {
    'train': datasets.ImageFolder(root='./train', transform=image_transforms['train']),
    'validation': datasets.ImageFolder(root='./validation', transform=image_transforms['validation']),
    'test': datasets.ImageFolder(root='./test', transform=image_transforms['test'])
}

# create a data loader instances

batch_size = 40

train_data = loader.DataLoader(data['train'], batch_size=batch_size, shuffle=True)
validation_data = loader.DataLoader(data['validation'], batch_size=batch_size, shuffle=True)
test_data = loader.DataLoader(data['test'], batch_size=batch_size, shuffle=True)

FileNotFoundError: ignored

In [ ]:
train_data_size = len(data['train'])
print(f"Number of Images in the Training Dataset size is {train_data_size}")

validation_data_size = len(data['validation'])
print(f"Number of Images in the Validation Dataset size is {validation_data_size}")

test_data_size =  len(data['test'])
print(f"Number of Images in the Test Dataset size is {test_data_size}")

In [ ]:
import torch.optim as optim
loss_func = nn.NLLLoss()
optimizer = optim.AdamW(model.parameters())

In [ ]:
import time

def train_and_validate(model, loss_criterion, optimizer, epochs=30):
  device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
  start = time.time()
  history = []
  best_acc = 0.0

  for epoch in range(1,epochs+1):
    epoch_start = time.time()
    print(f'Epoch : {epoch}/{epochs}')
    model.train()
    train_loss = 0.0
    train_acc = 0.0

    valid_loss = 0.0
    valid_acc  = 0.0

    for i, (inputs, labels) in enumerate(train_data):
      # all data, models, parameters etc have to be in the same device
      inputs = inputs.to(device)
      labels = labels.to(device)

      # Clean existing gradients
      optimizer.zero_grad()

      # Forward pass - compute outputs on input data using the model
      outputs = model(inputs)

      # Compute loss
      loss = loss_criterion(outputs, labels)

      # Backpropagate the gradients
      loss.backward()

      # Update the parameters
      optimizer.step()

      # Compute the total loss for the batch and it to train_loss
      train_loss += loss.item() * inputs.size(0)

      # Compute the accuracy
      ret, predictions = torch.max(outputs.data,1)
      correct_counts = predictions.eq(labels.data.view_as(predictions))

      # Convert correct_counts to float and then compute the mean
      acc = torch.mean(correct_counts.type(torch.FloatTensor))

      # Compute total accuracy in the whole batch and add to train_acc
      train_acc += acc.item() * inputs.size(0)
      print(f'Batch number: {i+1}, Training: Loss: {loss.item()}, Accuracy: {acc.item()}')

    with torch.no_grad():
      model.eval()
      for j, (inputs, labels) in enumerate(validation_data):
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass - compute outputs on input data using the model
        outputs = model(inputs)

        # Compute loss
        loss = loss_criterion(outputs, labels)

        # Compute the total loss for  the batch and add it to valid_loss
        valid_loss += loss.item() * inputs.size(0)

        # Calculate validation accuracy
        ret, predictions = torch.max(outputs.data, 1)
        correct_prediction_counts = predictions.eq(labels.data.view_as(predictions))

        # Convert correct_prediction_counts to float and then compute the mean
        acc = torch.mean(correct_prediction_counts.type(torch.FloatTensor))

        # Compute total accuracy in the whole batch and add to valid_acc
        valid_acc +=acc.item() * inputs.size(0)

      avg_train_loss = train_loss/train_data_size
      avg_train_acc = train_acc/train_data_size

      avg_valid_loss = valid_loss/validation_data_size
      avg_valid_acc = valid_acc/validation_data_size

      history.append([avg_train_loss, avg_valid_loss, avg_train_acc, avg_valid_acc])

      epoch_end = time.time()

      print(f'Epoch : {epoch}, Training: Loss: f{avg_train_loss}, Accuracy: {avg_train_acc*100}%, \n\t\tValidation : Loss : {avg_valid_loss}, Accuracy: {avg_valid_acc*100}%, Time: {epoch_end-epoch_start}s')

      # Save if the model has best accuracy till now

      torch.save(model.state_dict(), f'model_{epoch}.pth')

  return model, history

In [ ]:
num_epochs = 50
trained_model, history = train_and_validate(model, loss_func, optimizer, num_epochs)

In [ ]:
import matplotlib.pyplot as plt

losses1=[]
acc1=[]
losses2=[]
acc2=[]

for i in range(1,num_epochs+1):
  losses1.append(history[i-1][0])
  losses2.append(history[i-1][1])
  acc1.append(history[i-1][2])
  acc2.append(history[i-1][3])

plt.figure()
plt.plot(range(1,num_epochs+1),losses1)
plt.plot(range(1,num_epochs+1),losses2)
plt.title("Loss values for different epochs")
plt.legend(["Training Loss","Validation Loss"])

plt.figure()
plt.plot(range(1,num_epochs+1),acc1)
plt.plot(range(1,num_epochs+1),acc2)
plt.title("Accuracy values for different epochs")
plt.legend(["Training Accuracy","Validation Accuracy"])


acc1 = np.array(acc1)
acc2 = np.array(acc2)

print(f"The obtained maximum training accuracy is {np.max(acc1)}","+", f" at {np.argmax(acc1)+1}th epoch")
print(f"The obtained maximum validation accuracy is {np.max(acc2)}","+", f" at {np.argmax(acc2)+1}th epoch")

In [ ]:
def computeModelAccuracy(model, loss_criterion):
  device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
  test_acc = 0.0
  test_loss = 0.0

  with torch.no_grad():
    # Set to evaluation mode
    model.eval()
    for i, (inputs, labels) in enumerate(test_data):
      inputs = inputs.to(device)
      labels = labels.to(device)
      outputs = model(inputs)

      # Compute loss
      loss = loss_criterion(outputs, labels)

      # Compute the toal loss item
      test_loss += loss.item() * inputs.size(0)

      ret, predictions = torch.max(outputs.data, 1)
      correct_counts = predictions.eq(labels.data.view_as(predictions))
      acc = torch.mean(correct_counts.type(torch.FloatTensor))
      test_acc +=acc.item() * inputs.size(0)

      print(f'Test Batch number: {i+1}, Test: Loss: {loss.item()}, Accuracy: {acc.item()}')

      # Find average test loss and test accuracy
      avg_test_loss = test_loss/test_data_size
      avg_test_acc = test_acc/test_data_size

      print(f'Test accuracy: {avg_test_acc}')

In [ ]:
computeModelAccuracy(model, loss_func)

In [ ]:
from PIL import Image
import requests
import matplotlib.pyplot as plt

index_to_class = {v: k for k, v in data['train'].class_to_idx.items()}
print (index_to_class)

def makePrediction(model, url):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    transform = image_transforms['test']

    test_image = Image.open(requests.get(url, stream=True).raw)

    plt.figure()
    plt.imshow(test_image)

    test_image_tensor = transform(test_image)
    test_image_tensor = test_image_tensor.view(1, 3, 224, 224).to(device)
    #test_image_tensor = test_image_tensor.view(1, 3, 300, 300).to(device)
    with torch.no_grad():
        model.eval()
        out = model(test_image_tensor)
        ps = torch.exp(out)

        topk, topclass = ps.topk(4, dim=1)
        for i in range(4):
            print(f"Prediction {i+1} : {index_to_class[topclass.cpu().numpy()[0][i]]}, Score: {topk.cpu().numpy()[0][i] * 100}%")
        print("     ")

In [ ]:
# Choose only one code block to use one of these pretrained models to see and display predictions

# model_new = models.efficientnet_b3(pretrained=True)
# for param in model_new.parameters():
#   param.requires_grad = False
# my_classifier_inputs = model_new.classifier[1].in_features
# model_new.classifier[1] = nn.Sequential(
#     nn.Linear(my_classifier_inputs, 1024),
#     nn.ReLU(inplace=True),
#     nn.Linear(1024, 5),
#     nn.Dropout(0.3),
#     nn.LogSoftmax(dim=1)
# )
# model_new.load_state_dict(torch.load('model_29.pth'))
#model_new = model_new.to(device)



model_new = models.densenet161(pretrained=True)
for param in model.parameters():
  param.requires_grad = False
classifier_inputs = model_new.classifier.in_features
model_new.classifier = nn.Sequential(
    nn.Linear(classifier_inputs, 1024),
    nn.ReLU(inplace=True),

    nn.Linear(1024, 4),
    nn.Dropout(0.3),
    nn.LogSoftmax(dim=1)
)
model_new.load_state_dict(torch.load('model_49.pth'))
model_new = model_new.to(device)



# model_new = models.resnet152(pretrained=True)
# for param in model_new.parameters():
#  param.requires_grad = False
# fc_inputs = model_new.fc.in_features
# model_new.fc = nn.Sequential(
#    nn.Linear(fc_inputs, 1024),
#    nn.ReLU(inplace=True),
#    nn.Linear(1024, 4),
#    nn.Dropout(0.3),
#    nn.LogSoftmax(dim=1)
# )
# model_new.load_state_dict(torch.load('model_49.pth'))
# model_new = model_new.to(device)


makePrediction(model_new, 'https://www.spandidos-publications.com/article_images/ol/8/2/OL-08-02-0781-g01.jpg')

makePrediction(model_new, 'https://mayfieldclinic.com/images/meningioma-care-scan.jpg')

makePrediction(model_new,'https://i.ytimg.com/vi/G2YsuVzg-Gg/hqdefault.jpg')